# 01 — Dataset provenance & first audit
**Level: beginner.** Never assume a public CICFlowMeter CSV has the same semantics as NoHackFlow. Work only with approved labeled canonical data, and record where it came from.

**Goal:** inspect file size, labels, columns, types, duplicates and missing values without training. **No dataset or metric is bundled.**

In [ ]:
from pathlib import Path
import os
import pandas as pd
from nohack.features import FEATURE_NAMES
path = Path(os.environ.get("NOHACK_DATASET", "data/interim/labeled_nohackflow.csv"))
if not path.is_file():
    raise FileNotFoundError(f"Provide your OWN canonical data with NOHACK_DATASET: {path}")
df = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
print("Rows:", len(df), "Columns:", len(df.columns))
print("Dataset filename:", path.name)  # Do not publish private local paths
print(df.dtypes.value_counts().to_string())

In [ ]:
required = set(FEATURE_NAMES) | {"label", "group_id"}
print("Missing canonical fields:", sorted(required - set(df.columns)))
print("Unexpected fields:", sorted(set(df.columns) - required - {"attack_family"}))
print("Label counts:\n", df["label"].value_counts(dropna=False) if "label" in df else "not supplied")
print("Null counts (top 10):\n", df.isna().sum().sort_values(ascending=False).head(10))
print("Exact duplicated rows:", int(df.duplicated().sum()))

In [ ]:
from nohack.datasets.prepare import validate_canonical_table
clean = validate_canonical_table(df)
print("Schema validated. Number of independent groups:", clean.group_id.nunique())

### Interpret the output
Ask: Is each label independently verified? Do group IDs identify sessions/scenarios rather than each row? Do feature formulas match the Windows collector? If not, stop before training. Never upload IP addresses or identifiable captures to public Kaggle.